# DLAV Project - Phase 1

This notebook is the single entry point for the project in both local VS Code and Google Colab.
It keeps the current `src/` training pipeline intact while standardizing the project paths, data flow, and output locations.


## Environment Setup

The next cell detects whether the notebook is running locally or in Colab.
On Colab, it clones the repository into `/content/dlav-project` when needed and then defines the project paths from there.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/math707/dlav-project.git'
COLAB_PROJECT_DIR = Path('/content/dlav-project')


def is_running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def find_project_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    return None


def ensure_colab_repo(repo_url: str, target_dir: Path) -> Path:
    git_dir = target_dir / '.git'

    if git_dir.is_dir():
        print(f'Updating repository in {target_dir}...')
        subprocess.check_call(['git', '-C', str(target_dir), 'pull', '--ff-only'])
        return target_dir

    if target_dir.exists():
        if (target_dir / 'src').is_dir() and (target_dir / 'notebooks').is_dir():
            print(f'Using existing project directory in {target_dir}...')
            return target_dir
        raise FileExistsError(
            f'{target_dir} exists but is not a Git repository or recognized project root.'
        )

    print(f'Cloning repository into {target_dir}...')
    subprocess.check_call(['git', 'clone', repo_url, str(target_dir)])
    return target_dir


IN_COLAB = is_running_in_colab()
project_root = find_project_root(Path.cwd())

if IN_COLAB:
    if project_root is None:
        project_root = ensure_colab_repo(REPO_URL, COLAB_PROJECT_DIR)
    elif project_root == COLAB_PROJECT_DIR and (COLAB_PROJECT_DIR / '.git').is_dir():
        print(f'Updating repository in {COLAB_PROJECT_DIR}...')
        subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
    os.chdir(project_root)
elif project_root is None:
    raise FileNotFoundError(
        'Could not find the project root. Open the notebook from inside the repository.'
    )

PROJECT_ROOT = project_root.resolve()
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SRC_DIR = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data'
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'
TEST_DIR = DATA_DIR / 'test_public'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUTS_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUTS_DIR / 'submissions'

for path in (DATA_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR, CHECKPOINT_DIR, SUBMISSION_DIR):
    path.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / 'phase1_model.pth'
SUBMISSION_PATH = SUBMISSION_DIR / 'submission_phase1.csv'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Environment: {"Google Colab" if IN_COLAB else "Local"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Data directory: {DATA_DIR}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Submission directory: {SUBMISSION_DIR}')


## Data Preparation

The dataset archives are stored and extracted under `data/`.
Missing data is downloaded only when needed, and already extracted folders are reused as-is.


In [ ]:
import importlib.util
import zipfile


DATASETS = {
    'train': {
        'file_id': '1YkGwaxBKNiYL2nq--cB6WMmYGzRmRKVr',
        'zip_name': 'dlav_train.zip',
        'target_dir': TRAIN_DIR,
    },
    'val': {
        'file_id': '1wtmT_vH9mMUNOwrNOMFP6WFw6e8rbOdu',
        'zip_name': 'dlav_val.zip',
        'target_dir': VAL_DIR,
    },
    'test_public': {
        'file_id': '1G9xGE7s-Ikvvc2-LZTUyuzhWAlNdLTLV',
        'zip_name': 'dlav_test_public.zip',
        'target_dir': TEST_DIR,
    },
}


def has_pkl_files(directory: Path) -> bool:
    return directory.is_dir() and any(directory.glob('*.pkl'))


def ensure_gdown():
    if importlib.util.find_spec('gdown') is None:
        if not IN_COLAB:
            raise ImportError(
                'gdown is required only for automatic dataset download. '
                'Install it locally or place the archives/files manually under data/.'
            )
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown
    return gdown


def ensure_dataset(name: str, file_id: str, zip_name: str, target_dir: Path) -> None:
    zip_path = DATA_DIR / zip_name

    if has_pkl_files(target_dir):
        print(f'{name}: found extracted files in {target_dir.relative_to(PROJECT_ROOT)}')
        return

    if not zip_path.exists():
        gdown = ensure_gdown()
        download_url = f'https://drive.google.com/uc?id={file_id}'
        print(f'{name}: downloading {zip_name}...')
        gdown.download(download_url, str(zip_path), quiet=False)
    else:
        print(f'{name}: found archive {zip_name}, skipping download.')

    print(f'{name}: extracting into {DATA_DIR.relative_to(PROJECT_ROOT)}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)

    if not has_pkl_files(target_dir):
        raise FileNotFoundError(f'{name}: no .pkl files found in {target_dir}')

    print(f'{name}: ready in {target_dir.relative_to(PROJECT_ROOT)}')


for dataset_name, dataset_config in DATASETS.items():
    ensure_dataset(dataset_name, **dataset_config)


## Imports and Runtime Configuration

We keep using the code from `src/` and configure a small set of notebook-level hyperparameters here.


In [ ]:
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from src.dataset import DrivingDataset
from src.logger import Logger
from src.model import DrivingPlanner
from src.train import train

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if (IN_COLAB or os.name != 'nt') else 0
PIN_MEMORY = DEVICE.type == 'cuda'

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 50


def sorted_pkl_files(directory: Path) -> list[Path]:
    return sorted(directory.glob('*.pkl'), key=lambda path: int(path.stem))


print(f'Device: {DEVICE}')
print(f'num_workers: {NUM_WORKERS}')


## Quick Data Check

Before training, we inspect a few random training examples.


In [ ]:
train_sample_files = sorted_pkl_files(TRAIN_DIR)
if not train_sample_files:
    raise FileNotFoundError(f'No training files found in {TRAIN_DIR}')

num_examples = min(4, len(train_sample_files))
sample_paths = random.sample(train_sample_files, k=num_examples)

samples = []
for sample_path in sample_paths:
    with open(sample_path, 'rb') as file:
        samples.append(pickle.load(file))

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, sample in zip(axes, samples):
    axis.imshow(sample['camera'])
    axis.axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, sample in zip(axes, samples):
    axis.plot(sample['sdc_history_feature'][:, 0], sample['sdc_history_feature'][:, 1], 'o-', color='gold', label='Past')
    axis.plot(sample['sdc_future_feature'][:, 0], sample['sdc_future_feature'][:, 1], 'o-', color='green', label='Future')
    axis.legend()
    axis.axis('equal')
plt.tight_layout()
plt.show()


## Datasets and Dataloaders

Paths now point explicitly to `data/train`, `data/val`, and `data/test_public`.


In [ ]:
train_files = sorted_pkl_files(TRAIN_DIR)
val_files = sorted_pkl_files(VAL_DIR)

if not train_files:
    raise FileNotFoundError(f'No training files found in {TRAIN_DIR}')
if not val_files:
    raise FileNotFoundError(f'No validation files found in {VAL_DIR}')

train_dataset = DrivingDataset(train_files)
val_dataset = DrivingDataset(val_files)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')


## Model and Training

The model architecture remains unchanged for now; only the project organization around it is cleaned up.


In [ ]:
model = DrivingPlanner()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
logger = Logger()

train(model, train_loader, val_loader, optimizer, logger, num_epochs=NUM_EPOCHS)


## Save Checkpoint

Checkpoints are saved in `outputs/checkpoints/`.


In [ ]:
torch.save(model.state_dict(), CHECKPOINT_PATH)
print(f'Model saved to {CHECKPOINT_PATH}')


## Qualitative Validation Plots

We compare predicted and ground-truth future trajectories on a validation batch.


In [ ]:
val_batch = next(iter(val_loader))
model = model.to(DEVICE)

camera = val_batch['camera'].to(DEVICE)
history = val_batch['history'].to(DEVICE)
future = val_batch['future'].to(DEVICE)

model.eval()
with torch.no_grad():
    pred_future = model(camera, history)

camera_np = camera.cpu().numpy()
history_np = history.cpu().numpy()
future_np = future.cpu().numpy()
pred_future_np = pred_future.cpu().numpy()

num_examples = min(4, len(camera_np))
selected_indices = random.sample(range(len(camera_np)), k=num_examples)

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, index in zip(axes, selected_indices):
    axis.imshow(camera_np[index].transpose(1, 2, 0) / 255.0)
    axis.axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, num_examples, figsize=(4 * num_examples, 4))
axes = np.atleast_1d(axes)
for axis, index in zip(axes, selected_indices):
    axis.plot(history_np[index, :, 0], history_np[index, :, 1], 'o-', color='gold', label='Past')
    axis.plot(future_np[index, :, 0], future_np[index, :, 1], 'o-', color='green', label='Future')
    axis.plot(pred_future_np[index, :, 0], pred_future_np[index, :, 1], 'o-', color='red', label='Predicted')
    axis.legend()
    axis.axis('equal')
plt.tight_layout()
plt.show()


## Submission Generation

The test set contains no future trajectory labels. We run inference once and write the submission to `outputs/submissions/`.


In [ ]:
with open(TEST_DIR / '0.pkl', 'rb') as file:
    test_sample = pickle.load(file)
print(test_sample.keys())


In [ ]:
test_files = sorted_pkl_files(TEST_DIR)
if not test_files:
    raise FileNotFoundError(f'No test files found in {TEST_DIR}')

test_dataset = DrivingDataset(test_files, test=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=250,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

model.eval()
all_plans = []
with torch.no_grad():
    for batch in test_loader:
        camera = batch['camera'].to(DEVICE)
        history = batch['history'].to(DEVICE)
        pred_future = model(camera, history)
        all_plans.append(pred_future.cpu().numpy()[..., :2])

all_plans = np.concatenate(all_plans, axis=0)

total_samples, timesteps, dimensions = all_plans.shape
flattened_plans = all_plans.reshape(total_samples, timesteps * dimensions)

submission = pd.DataFrame(flattened_plans)
submission.insert(0, 'id', np.arange(total_samples))

column_names = ['id']
for timestep in range(1, timesteps + 1):
    column_names.append(f'x_{timestep}')
    column_names.append(f'y_{timestep}')
submission.columns = column_names

submission.to_csv(SUBMISSION_PATH, index=False)

print(f'Submission saved to {SUBMISSION_PATH}')
print(f'Submission shape: {submission.shape}')
